### Setting

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
import os
import pandas as pd

import torch
import torch.nn as nn

#import gensim
#from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk

import re
import numpy as np

from sklearn.preprocessing import LabelEncoder

import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import torch.nn.functional as F

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

from collections import Counter

from transformers import BertTokenizer, BertModel

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available!")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

### Data Loader

In [ ]:
cd gdrive/MyDrive/Master degree/24T3/DATA5011/modelling_final/Datasets/ver8_10k

In [ ]:
I_plus_train = pd.read_parquet('I_plus_train_10k_ver8.parquet')
I_minus_train = pd.read_parquet('I_minus_train_10k_ver8.parquet')

I_plus_val = pd.read_parquet('I_plus_val_10k_ver8.parquet')
I_minus_val = pd.read_parquet('I_minus_val_10k_ver8.parquet')

test = pd.read_parquet('test_10k_ver8.parquet')

In [ ]:
I_plus_train

In [ ]:
# Initialize the BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Make sure the model is in evaluation mode
model.eval()

# Convert the 'TrackArtist' column into BERT embeddings
def get_bert_embedding(text):
    # Tokenize and encode the input text
    #inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=10, padding=True)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=32, padding=True)

    # Get the [CLS] token embeddings
    with torch.no_grad():  # Disable gradient calculation for efficiency
        outputs = model(**inputs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # Extract the [CLS] token representation

    return cls_embedding.squeeze().numpy()  # Convert to numpy for easier storage

In [ ]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['album_Bert_embedding'] = I_plus_train['album_name'].apply(get_bert_embedding)
I_minus_train['album_Bert_embedding'] = I_minus_train['album_name'].apply(get_bert_embedding)
I_plus_val['album_Bert_embedding'] = I_plus_val['album_name'].apply(get_bert_embedding)
I_minus_val['album_Bert_embedding'] = I_minus_val['album_name'].apply(get_bert_embedding)
test['album_Bert_embedding'] = test['album_name'].apply(get_bert_embedding)
#music_pool['album_Bert_embedding'] = music_pool['lemmatized_album_name'].apply(get_bert_embedding)

In [ ]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['track_Bert_embedding'] = I_plus_train['track_name'].apply(get_bert_embedding)
I_minus_train['track_Bert_embedding'] = I_minus_train['track_name'].apply(get_bert_embedding)
I_plus_val['track_Bert_embedding'] = I_plus_val['track_name'].apply(get_bert_embedding)
I_minus_val['track_Bert_embedding'] = I_minus_val['track_name'].apply(get_bert_embedding)
test['track_Bert_embedding'] = test['track_name'].apply(get_bert_embedding)
#music_pool['track_Bert_embedding'] = music_pool['lemmatized_track_name'].apply(get_bert_embedding)

In [ ]:
test

In [ ]:
I_plus_train.to_parquet("I_plus_train.parquet", index=False)
I_minus_train.to_parquet("I_minus_train.parquet", index=False)
I_plus_val.to_parquet("I_plus_val.parquet", index=False)
I_minus_val.to_parquet("I_minus_val.parquet", index=False)
test.to_parquet("test.parquet", index=False)
#music_pool.to_parquet("music_pool.parquet", index=False)

In [ ]:
# Check if all embeddings have the same shape
unique_shapes = I_plus_train['album_Bert_embedding'].apply(lambda x: x.shape).unique()
print("Unique shapes in 'bert_embedding' column:", unique_shapes)


### Track + Artist + Album

In [ ]:
I_plus_train['AlbumTrackArtist'] = I_plus_train['lemmatized_album_name'] + " - " + I_plus_train['lemmatized_track_name'] + " by " + I_plus_train['artist_name']
I_minus_train['AlbumTrackArtist'] = I_minus_train['lemmatized_album_name'] + " - " + I_minus_train['lemmatized_track_name'] + " by " + I_minus_train['artist_name']
I_plus_val['AlbumTrackArtist'] = I_plus_val['lemmatized_album_name'] + " - " + I_plus_val['lemmatized_track_name'] + " by " + I_plus_val['artist_name']
I_minus_val['AlbumTrackArtist'] = I_minus_val['lemmatized_album_name'] + " - " + I_minus_val['lemmatized_track_name'] + " by " + I_minus_val['artist_name']

In [ ]:
music_pool['AlbumTrackArtist'] = music_pool['lemmatized_album_name'] + " - " + music_pool['lemmatized_track_name'] + " by " + music_pool['artist_name']

In [ ]:
# Apply the BERT embedding function to the 'TrackArtist' column
I_plus_train['AlbumTrackArtist_Bert_embedding'] = I_plus_train['AlbumTrackArtist'].apply(get_bert_embedding)
I_minus_train['AlbumTrackArtist_Bert_embedding'] = I_minus_train['AlbumTrackArtist'].apply(get_bert_embedding)
I_plus_val['AlbumTrackArtist_Bert_embedding'] = I_plus_val['AlbumTrackArtist'].apply(get_bert_embedding)
I_minus_val['AlbumTrackArtist_Bert_embedding'] = I_minus_val['AlbumTrackArtist'].apply(get_bert_embedding)


In [ ]:
music_pool['AlbumTrackArtist_Bert_embedding'] = music_pool['AlbumTrackArtist'].apply(get_bert_embedding)

In [ ]:
# Check if all embeddings have the same shape
unique_shapes = I_plus_train['AlbumTrackArtist_Bert_embedding'].apply(lambda x: x.shape).unique()
print("Unique shapes in 'bert_embedding' column:", unique_shapes)

In [ ]:
I_plus_train.to_pickle("I_plus_train_with_bert_albumtrackartist.pkl")
I_minus_train.to_pickle("I_minus_train_with_bert_albumtrackartist.pkl")
I_plus_val.to_pickle("I_plus_val_with_bert_albumtrackartist.pkl")
I_minus_val.to_pickle("I_minus_val_with_bert_albumtrackartist.pkl")
music_pool.to_pickle("music_pool_with_bert_albumtrackartist.pkl")